# 01 - Discovery + Validation
Part of the Public Webcam Discovery System.

Runs DirectoryAgent (Tier 1 sources from SOURCES.md) to collect camera candidates,
then ValidationAgent to confirm live feeds.

In [ ]:
# Environment setup (run once)
import subprocess, sys, os
from pathlib import Path

IN_COLAB     = "google.colab" in sys.modules
IN_SAGEMAKER = os.environ.get("SM_TRAINING_ENV") is not None

if IN_COLAB:
    if not Path("webcam-discovery").exists():
        subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/webcam-discovery"], check=True)
    os.chdir("webcam-discovery")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[notebooks]", "-q"], check=True)

from webcam_discovery.config import settings
settings.ensure_dirs()
print("Ready")

## Step 2 - Build environment

Install the package with dev dependencies so the pipeline and tests are available.

In [ ]:
!pip install -e ".[dev]" -q

## Step 3 - Run DirectoryAgent (Tier 1)

Crawls all Tier 1 sources from `SOURCES.md`, resolves direct feed URLs via
`FeedExtractionSkill`, and writes results to `candidates/candidates.jsonl`.

Changes since last review:
- Sources are now read from **SOURCES.md** at runtime (no hardcoded lists).
- **All 5 tiers** are supported; `--tier N` crawls tiers 1-N.
- `FeedExtractionSkill` now uses **finditer** (not early-return), so listing pages
  with multiple embedded cameras surface all links.
- Shallow listing-page URLs (path depth < 3) that yielded no stream are **dropped**.
- Each embedded link from a page becomes a **sub-candidate**.

In [ ]:
!python scripts/run_discovery.py --tier 1 --output candidates/candidates.jsonl

## Step 4 - Inspect candidates.jsonl

Preview candidate counts broken down by URL type:

| Type | Meaning |
|------|---------|
| `direct-stream` | HLS / MJPEG / MP4 - confirmed stream URL |
| `youtube-embed` | YouTube nocookie embed |
| `embed-page`    | Third-party player embed page |
| `html-page`     | Camera embed page (ValidationAgent will classify) |

In [ ]:
import json, re
from pathlib import Path
from collections import Counter

output_path = Path("candidates/candidates.jsonl")

if not output_path.exists():
    print("candidates.jsonl not found - did the crawler run successfully?")
else:
    lines = output_path.read_text().splitlines()
    candidates = [json.loads(l) for l in lines if l.strip()]

    _stream_ext = re.compile(r'\.(m3u8|mjpg|mjpeg|mp4)(\?|$)', re.IGNORECASE)
    _youtube_re = re.compile(r'youtube(?:-nocookie)?\.com/embed/', re.IGNORECASE)
    _embed_re   = re.compile(r'/embed[/\?]|embed\.', re.IGNORECASE)

    def url_type(url):
        if _stream_ext.search(url): return 'direct-stream'
        if _youtube_re.search(url): return 'youtube-embed'
        if _embed_re.search(url):   return 'embed-page'
        return 'html-page'

    types = Counter(url_type(c['url']) for c in candidates)

    print(f"Total candidates: {len(candidates)}")
    print(f"  direct-stream  : {types['direct-stream']:>4}  (HLS / MJPEG / MP4)")
    print(f"  youtube-embed  : {types['youtube-embed']:>4}  (YouTube nocookie embed)")
    print(f"  embed-page     : {types['embed-page']:>4}  (third-party player embed)")
    print(f"  html-page      : {types['html-page']:>4}  (camera embed page)")
    print()

    for c in candidates[:10]:
        t = url_type(c['url'])
        label = c.get('label', '-')
        city  = c.get('city', '-')
        ctry  = c.get('country', '-')
        print(f"  [{t:<14}]  {c['url']}")
        print(f"               label={label}  city={city}, {ctry}")
        print()

## Step 5 - Validate candidates

`ValidationAgent` deep-probes each candidate URL to confirm it is a **live video feed**,
not just a reachable page.

### What changed (validation overhaul)

| Before | After |
|--------|-------|
| 5 s total timeout -> most streams timed out | `connect=10 s, read=25 s` - generous for slow camera servers |
| 500 concurrent requests, no semaphore | `asyncio.Semaphore(50)` limits concurrent probes |
| HTML 200 OK = medium/live regardless | Deep HTML scan for live-player patterns (HLS.js, JW Player, Video.js) |
| Candidates dropped if city unknown | Coordinates are **Optional** - 4-strategy fallback geocoding |
| Sequential geopy calls | `ThreadPoolExecutor(20)` runs geo lookups in parallel threads |
| `min_legitimacy=medium` - only 4 passed | `min_legitimacy=low` - all confirmed live feeds accepted |

### Geocoding fallback chain (for cameras without city)

1. City + country via Nominatim
2. Label text via Nominatim (e.g. "Times Square NYC")
3. IP geolocation of camera hostname via ip-api.com
4. Country-center via Nominatim

### Legitimacy grades

| Grade | How assigned |
|-------|--------------|
| **high** | HLS playlist has `#EXTM3U`, MJPEG has `multipart/x-mixed-replace`, or HTML has live-player pattern |
| **medium** | HTML has `<video>` tag or webcam/live keywords |
| **low** | 200 OK but no video detected - included but flagged |

Results -> `candidates/validated.jsonl`

In [ ]:
!python scripts/run_validation.py --input candidates/candidates.jsonl --output candidates/validated.jsonl

## Step 6 - Inspect validated cameras

Pipeline funnel, feed-type breakdown, and first 10 confirmed camera records.

In [ ]:
import json, re
from pathlib import Path
from collections import Counter

validated_path  = Path("candidates/validated.jsonl")
candidates_path = Path("candidates/candidates.jsonl")

if not validated_path.exists():
    print("validated.jsonl not found - did validation run successfully?")
else:
    records = [json.loads(l) for l in validated_path.read_text().splitlines() if l.strip()]
    n_cands = sum(1 for l in candidates_path.read_text().splitlines() if l.strip()) \
              if candidates_path.exists() else '?'

    located   = [r for r in records if r.get('latitude') is not None]
    unlocated = [r for r in records if r.get('latitude') is None]

    _stream_re = re.compile(r'\.(m3u8|mjpg|mjpeg|mp4)(\?|$)', re.IGNORECASE)
    _yt_re     = re.compile(r'youtube(?:-nocookie)?\.com/embed/', re.IGNORECASE)

    def feed_quality(r):
        url = r['url']
        if _stream_re.search(url):            return 'direct-stream'
        if _yt_re.search(url):                return 'youtube-embed'
        if r['feed_type'] in ('HLS','MJPEG'): return 'confirmed-stream'
        if r['legitimacy_score'] == 'high':   return 'live-player-detected'
        if r['legitimacy_score'] == 'medium': return 'probable-live'
        return 'low-confidence'

    quality = Counter(feed_quality(r) for r in records)

    print("-- Pipeline funnel ------------------------------------------")
    print(f"  Candidates (discovery)      : {n_cands}")
    print(f"  Validated records           : {len(records)}")
    print(f"    with coordinates          : {len(located)}")
    print(f"    without coords (included) : {len(unlocated)}")
    print()
    print("-- Live-feed quality breakdown --------------------------------")
    for q, n in quality.most_common():
        print(f"  {q:<24} : {n}")
    print()
    print("-- By feed type -----------------------------------------------")
    for ft, n in Counter(r['feed_type'] for r in records).most_common():
        print(f"  {ft:<16} : {n}")
    print()
    print("-- By legitimacy ----------------------------------------------")
    for ls, n in Counter(r['legitimacy_score'] for r in records).most_common():
        print(f"  {ls:<8} : {n}")
    print()
    print("-- First 10 (highest legitimacy first) ------------------------")
    top = sorted(records,
                 key=lambda r: {'high':2,'medium':1,'low':0}.get(r['legitimacy_score'],0),
                 reverse=True)
    for r in top[:10]:
        coords = f"{r['latitude']:.3f},{r['longitude']:.3f}" if r.get('latitude') else 'no-coords'
        print(f"  [{r['feed_type']:<14}] [{r['legitimacy_score']:<6}] [{coords}]")
        print(f"    {r['url']}")
        label   = r.get('label', '-')
        city    = r.get('city', '-')
        country = r.get('country', '-')
        print(f"    {label} - {city}, {country}")
        print()